# Prediccion de Viralidad en YouTube
## Proyecto Final - Maquina de Aprendizaje 1
### Universidad de la Sabana | Metodologia CRISP-DM

---

**Dataset:** `global_youtube_creator_data_large.csv`  
**Variable objetivo:** `viralidad` (binaria: 1 = viral, 0 = no viral)  
**Modelos:** Regresion Logistica | Random Forest | Gradient Boosting  

---

## Configuracion del entorno

In [ ]:
import sys, os
# Agregar src/ al path para importar modulos del proyecto
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import joblib

from sklearn.pipeline        import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics         import classification_report, roc_auc_score

from src.data_loader        import load_raw_data, basic_profiling
from src.feature_engineering import build_viralidad, build_features, get_feature_columns
from src.preprocessing      import build_preprocessor, split_data, save_preprocessor
from src.modeling            import get_models, cross_validate_models, train_and_save
from src.evaluation          import (
    evaluate_on_test, plot_confusion_matrices, plot_roc_curves,
    plot_feature_importance, plot_cv_comparison, plot_metrics_heatmap
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RANDOM_STATE = 42
SAMPLE_SIZE  = 100_000
RAW_PATH     = '../global_youtube_creator_data_large.csv'
FIGURES_DIR  = '../reports/figures'

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Entorno configurado correctamente.')

---
# FASE 1 | COMPRENSION DEL NEGOCIO (Business Understanding)
---

## 1.1 Contexto del problema

YouTube es la plataforma de video mas grande del mundo, con mas de 500 horas de contenido subido cada minuto.  
Para los creadores de contenido y las marcas, entender **que hace que un video se vuelva viral** es de valor estrategico: permite optimizar el tipo de contenido, el idioma de publicacion, el momento de subida y la duracion del video.

## 1.2 Objetivo del negocio

> Desarrollar un modelo de clasificacion que prediga si un video de YouTube sera **viral** (alto engagement relativo) o **no viral**, a partir de caracteristicas contextuales y de engagement relativo del video.

## 1.3 Objetivo analitico

- Construir la variable objetivo `viralidad` a partir de las metricas de engagement absolutas (views, likes, comments, shares) usando un puntaje compuesto de rango percentil ponderado.
- Entrenar y comparar tres clasificadores: **Regresion Logistica**, **Random Forest** y **Gradient Boosting**.
- Evaluar el desempeno con validacion cruzada estratificada (5-fold) y sobre un conjunto de prueba separado.

## 1.4 Criterios de exito

| Metrica   | Umbral minimo aceptable |
|-----------|------------------------|
| ROC-AUC   | > 0.70                 |
| F1-Score  | > 0.65                 |
| Accuracy  | > 0.65                 |

## 1.5 Pregunta de investigacion

> *Con base en la eficiencia de engagement de un video (likes por vista, comentarios por vista, shares por vista) y sus caracteristicas contextuales (categoria, idioma, region, duracion, sentimiento, hora de publicacion), ¿es posible predecir si el video tendra un nivel de viralidad alto?*

---
# FASE 2 | COMPRENSION DE LOS DATOS (Data Understanding)
---

## 2.1 Carga de datos

In [ ]:
df_raw = load_raw_data(RAW_PATH, sample_size=SAMPLE_SIZE, random_state=RANDOM_STATE)
basic_profiling(df_raw)

## 2.2 Descripcion de variables

| Variable        | Tipo        | Descripcion                                         |
|-----------------|-------------|-----------------------------------------------------|
| timestamp       | datetime    | Fecha y hora de publicacion del video               |
| video_id        | categorica  | Identificador unico del video                       |
| category        | categorica  | Categoria del contenido (6 categorias)              |
| language        | categorica  | Idioma del video (5 idiomas)                        |
| region          | categorica  | Region de origen (5 regiones)                       |
| duration_sec    | numerica    | Duracion del video en segundos [60, 3600]           |
| views           | numerica    | Numero total de visualizaciones                     |
| likes           | numerica    | Numero total de likes                               |
| comments        | numerica    | Numero total de comentarios                         |
| shares          | numerica    | Numero total de compartidos                         |
| sentiment_score | numerica    | Puntaje de sentimiento de comentarios [-1, 1]       |
| ads_enabled     | binaria     | Si el video tiene publicidad habilitada (True/False) |

## 2.3 Analisis de valores nulos y calidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Nulos
nulls = df_raw.isnull().sum()
axes[0].barh(nulls.index, nulls.values, color='#C44E52')
axes[0].set_title('Valores Nulos por Columna', fontweight='bold')
axes[0].set_xlabel('Cantidad')

# Duplicados
n_dup = df_raw.duplicated().sum()
axes[1].bar(['Unicos', 'Duplicados'], [len(df_raw) - n_dup, n_dup],
             color=['#55A868', '#C44E52'])
axes[1].set_title(f'Registros Unicos vs Duplicados\n(Total: {len(df_raw):,})', fontweight='bold')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/calidad_datos.png', bbox_inches='tight')
plt.show()
print(f'Valores nulos: {nulls.sum()} | Duplicados: {n_dup}')

## 2.4 Distribucion de variables numericas de engagement

In [ ]:
engagement_cols = ['views', 'likes', 'comments', 'shares']
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, col in enumerate(engagement_cols):
    # Escala original
    axes[0, i].hist(df_raw[col], bins=50, color='#4C72B0', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(f'{col} (escala original)', fontsize=10)
    axes[0, i].set_xlabel('Valor')
    axes[0, i].set_ylabel('Frecuencia')

    # Escala log
    axes[1, i].hist(np.log1p(df_raw[col]), bins=50, color='#C44E52', alpha=0.8, edgecolor='white')
    axes[1, i].set_title(f'log1p({col})', fontsize=10)
    axes[1, i].set_xlabel('log1p(Valor)')
    axes[1, i].set_ylabel('Frecuencia')

fig.suptitle('Distribucion de Metricas de Engagement - Escala Original vs Logaritmica',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/distribucion_engagement.png', bbox_inches='tight')
plt.show()

print('\nObservacion: Las metricas siguen una distribucion de cola larga (tipo ley de potencia).')
print('La transformacion log1p simetriza la distribucion, lo cual es crucial para la')
print('construccion del puntaje de viralidad y para el rendimiento de la regresion logistica.')

## 2.5 Analisis de variables categoricas

In [ ]:
cat_cols = ['category', 'language', 'region']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, cat_cols):
    counts = df_raw[col].value_counts()
    ax.bar(counts.index, counts.values, color='#4C72B0', alpha=0.85)
    ax.set_title(f'Distribucion de {col}', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frecuencia')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Variables Categoricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/distribucion_categoricas.png', bbox_inches='tight')
plt.show()

## 2.6 Matriz de correlacion

In [ ]:
num_cols = ['views', 'likes', 'comments', 'shares', 'duration_sec', 'sentiment_score']
corr = df_raw[num_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlacion de Variables Numericas', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/correlacion.png', bbox_inches='tight')
plt.show()

print('\nHallazgos clave:')
print(f'  - Correlacion views-likes    : {corr.loc["views","likes"]:.3f}')
print(f'  - Correlacion views-comments : {corr.loc["views","comments"]:.3f}')
print(f'  - Correlacion views-shares   : {corr.loc["views","shares"]:.3f}')
print(f'  - Correlacion likes-comments : {corr.loc["likes","comments"]:.3f}')

## 2.7 Engagement promedio por categoria

In [ ]:
engagement_by_cat = df_raw.groupby('category')[engagement_cols].mean().sort_values('views', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
colors = sns.color_palette('muted', len(engagement_by_cat))

for ax, col in zip(axes.flat, engagement_cols):
    bars = ax.bar(engagement_by_cat.index, engagement_by_cat[col], color=colors)
    ax.set_title(f'Promedio de {col} por Categoria', fontweight='bold')
    ax.set_xlabel('Categoria')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Engagement Promedio por Categoria de Contenido', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/engagement_por_categoria.png', bbox_inches='tight')
plt.show()

---
# FASE 3 | PREPARACION DE LOS DATOS (Data Preparation)
---

## 3.1 Limpieza de datos

In [ ]:
df = df_raw.copy()

# Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()
print(f'Duplicados eliminados: {antes - len(df)}')

# Eliminar registros con views = 0 (no aportan informacion de engagement)
df = df[df['views'] > 0].reset_index(drop=True)
print(f'Registros con views=0 eliminados: {antes - len(df)}')
print(f'Dataset limpio: {len(df):,} registros')

## 3.2 Construccion de la variable objetivo: VIRALIDAD

### Metodologia

La variable `viralidad` se construye en 4 pasos:

1. **Transformacion logaritmica** (`log1p`) de views, likes, comments, shares para reducir el sesgo de la distribucion de cola larga.
2. **Rango percentil** de cada metrica transformada: ubica cada video en una posicion relativa (0 = menor, 1 = mayor) dentro del conjunto de datos.
3. **Puntaje compuesto ponderado**:

$$\text{engagement\_score} = 0.30 \cdot r_{views} + 0.25 \cdot r_{likes} + 0.20 \cdot r_{comments} + 0.25 \cdot r_{shares}$$

   Los pesos reflejan que las vistas (alcance) y los shares (propagacion) son los principales indicadores de viralidad, seguidos de likes y comentarios.

4. **Clasificacion binaria**: videos con `engagement_score >= percentil 50` son etiquetados como **virales** (1); los demas como **no virales** (0).

In [ ]:
df = build_viralidad(df, threshold_pct=50)

In [ ]:
# Visualizar el puntaje de engagement y el corte de viralidad
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribucion del puntaje
umbral = df[df['viralidad'] == 1]['engagement_score'].min()
axes[0].hist(df['engagement_score'], bins=60, color='#4C72B0', alpha=0.7, edgecolor='white')
axes[0].axvline(umbral, color='red', linestyle='--', lw=2, label=f'Umbral viral = {umbral:.3f}')
axes[0].set_title('Distribucion del Puntaje de Engagement', fontweight='bold')
axes[0].set_xlabel('engagement_score')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Balance de clases
class_counts = df['viralidad'].value_counts()
labels = ['No Viral (0)', 'Viral (1)']
colors = ['#C44E52', '#55A868']
bars = axes[1].bar(labels, class_counts.values, color=colors, alpha=0.85)
for bar, val in zip(bars, class_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}\n({val/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_title('Balance de Clases - Variable Viralidad', fontweight='bold')
axes[1].set_ylabel('Cantidad de Videos')

plt.suptitle('Variable Objetivo: Viralidad', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/viralidad_distribucion.png', bbox_inches='tight')
plt.show()

## 3.3 Ingenieria de caracteristicas

**Criterio de diseno:** Para evitar fuga de datos (*data leakage*), los modelos NO usan los valores brutos de views, likes, comments y shares como caracteristicas (ya que estos fueron usados para construir `viralidad` directamente). En cambio, se generan **tasas de engagement relativo** que miden la *eficiencia* de interaccion, que es conceptualmente distinta del alcance absoluto:

$$\text{likes\_per\_view} = \frac{\text{likes}}{\text{views}}$$

Un video puede tener pocos views pero una tasa de likes muy alta (contenido de nicho muy comprometido) o muchos views con una tasa de likes baja (contenido viral en alcance pero no en engagement cualitativo). Esta distincion hace el problema genuinamente interesante desde el punto de vista predictivo.

In [ ]:
df = build_features(df)

feature_info = get_feature_columns()
print('Caracteristicas por tipo:')
for tipo, cols in feature_info.items():
    print(f'  {tipo:12}: {cols}')

In [ ]:
# Visualizar tasas de engagement por clase
rate_cols = ['likes_per_view', 'comments_per_view', 'shares_per_view', 'engagement_rate']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col in zip(axes, rate_cols):
    for label, color in [(0, '#C44E52'), (1, '#55A868')]:
        subset = df[df['viralidad'] == label][col]
        ax.hist(np.log1p(subset), bins=40, alpha=0.6, color=color,
                label='No Viral' if label == 0 else 'Viral', density=True)
    ax.set_title(f'log1p({col})', fontsize=9, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.legend(fontsize=8)

plt.suptitle('Distribucion de Tasas de Engagement por Clase (Viral vs No Viral)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/engagement_rates_por_clase.png', bbox_inches='tight')
plt.show()

## 3.4 Particion train / test y preprocesamiento

In [ ]:
feature_info = get_feature_columns()
cat_cols = feature_info['categorical']
num_cols = feature_info['numerical']
bin_cols = feature_info['binary']
all_features = cat_cols + num_cols + bin_cols

X_train, X_test, y_train, y_test = split_data(
    df, feature_cols=all_features, target_col='viralidad',
    test_size=0.20, random_state=RANDOM_STATE
)

In [ ]:
# Construir y ajustar el preprocesador sobre los datos de entrenamiento
preprocessor = build_preprocessor(cat_cols, num_cols, bin_cols)
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

save_preprocessor(preprocessor, path='../models/preprocessor.joblib')

# Nombres de caracteristicas tras el preprocesamiento
ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names_proc = ohe_names + num_cols + bin_cols

print(f'\nDimensiones post-preprocesamiento:')
print(f'  X_train: {X_train_proc.shape}')
print(f'  X_test : {X_test_proc.shape}')
print(f'  Total features: {len(feature_names_proc)}')

In [ ]:
# Guardar datos procesados para reproducibilidad
cols_to_save = all_features + ['viralidad', 'engagement_score']
df[cols_to_save].to_csv('../data/processed/data_processed.csv', index=False)
print('Datos procesados guardados en data/processed/data_processed.csv')

---
# FASE 4 | MODELADO (Modeling)
---

## 4.1 Descripcion de modelos

### Regresion Logistica
Modelo lineal que estima la probabilidad de pertenencia a una clase mediante la funcion sigmoide. Sirve como **modelo de referencia (baseline)** por su simplicidad e interpretabilidad. Requiere que las variables esten escaladas (StandardScaler ya aplicado).

### Random Forest
Ensamble de arboles de decision entrenados con *bootstrap* sobre subconjuntos aleatorios de datos y caracteristicas (*bagging*). La prediccion final es el voto mayoritario. Robusto al ruido, maneja bien relaciones no lineales y provee importancia de caracteristicas.

### Gradient Boosting
Ensamble secuencial donde cada arbol corrige los errores del anterior minimizando una funcion de perdida mediante descenso de gradiente. `learning_rate` bajo (0.05) con `n_estimators` alto (200) equilibra bias-varianza. `subsample=0.8` agrega estocasticidad para mayor regularizacion.

## 4.2 Validacion cruzada estratificada (5-fold)

**StratifiedKFold** garantiza que cada fold tenga la misma proporcion de clases que el conjunto completo. Se reportan media y desviacion estandar de cada metrica para evaluar estabilidad del modelo.

In [ ]:
models = get_models(random_state=RANDOM_STATE)

print('Iniciando validacion cruzada estratificada (5-fold)...')
print('Nota: Gradient Boosting tarda mas debido a su naturaleza secuencial.\n')

cv_results = cross_validate_models(
    models, X_train_proc, y_train.values,
    cv_folds=5, random_state=RANDOM_STATE
)

In [ ]:
print('\nResumen de Validacion Cruzada:')
cols_show = ['ROC-AUC (val)', 'ROC-AUC std', 'F1 (val)', 'F1 std',
             'Accuracy (val)', 'Accuracy std', 'Precision (val)', 'Recall (val)',
             'Accuracy (train)', 'Tiempo (s)']
display(cv_results[cols_show].round(4))

In [ ]:
plot_cv_comparison(cv_results, save_path=f'{FIGURES_DIR}/cv_comparison.png')

## 4.3 Analisis de sobreajuste (overfitting)

La diferencia entre `Accuracy (train)` y `Accuracy (val)` en la validacion cruzada indica el grado de sobreajuste.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = np.arange(len(cv_results))
w = 0.35
colors_train = ['#4C72B0', '#55A868', '#C44E52']
colors_val   = ['#9FC8FF', '#AAE8C0', '#FF9999']

for i, (name, row) in enumerate(cv_results.iterrows()):
    ax.bar(x[i] - w/2, row['Accuracy (train)'], w, label=f'{name} (train)' if i==0 else '',
           color=colors_train[i], alpha=0.9)
    ax.bar(x[i] + w/2, row['Accuracy (val)'],   w, label=f'{name} (val)' if i==0 else '',
           color=colors_val[i],   alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(cv_results.index, rotation=10)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('Accuracy Train vs Validacion - Analisis de Sobreajuste', fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors_train[i], label=f'{name} train') for i, name in enumerate(cv_results.index)]
legend_elements += [Patch(facecolor=colors_val[i], label=f'{name} val') for i, name in enumerate(cv_results.index)]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/overfitting_analysis.png', bbox_inches='tight')
plt.show()

print('\nDiferencia Train - Val (indica sobreajuste):')
for name, row in cv_results.iterrows():
    diff = row['Accuracy (train)'] - row['Accuracy (val)']
    print(f'  {name:25}: {diff:+.4f}')

## 4.4 Entrenamiento final sobre todo el conjunto de entrenamiento

In [ ]:
trained_models = {}
for name, model in models.items():
    trained_models[name] = train_and_save(
        model, name, X_train_proc, y_train.values,
        models_dir='../models'
    )

---
# FASE 5 | EVALUACION (Evaluation)
---

## 5.1 Metricas en el conjunto de prueba

In [ ]:
test_results = evaluate_on_test(trained_models, X_test_proc, y_test.values)

In [ ]:
print('\nTabla de Metricas - Conjunto de Prueba:')
display(test_results.round(4))

In [ ]:
plot_metrics_heatmap(test_results, save_path=f'{FIGURES_DIR}/metrics_heatmap.png')

## 5.2 Matrices de confusion

In [ ]:
plot_confusion_matrices(trained_models, X_test_proc, y_test.values,
                         save_path=f'{FIGURES_DIR}/confusion_matrices.png')

## 5.3 Curvas ROC

In [ ]:
plot_roc_curves(trained_models, X_test_proc, y_test.values,
                save_path=f'{FIGURES_DIR}/roc_curves.png')

## 5.4 Importancia de caracteristicas

In [ ]:
for name, model in trained_models.items():
    plot_feature_importance(
        model, feature_names_proc, name, top_n=15,
        save_path=f'{FIGURES_DIR}/feature_importance.png'
    )

## 5.5 Comparacion global y seleccion del mejor modelo

In [ ]:
# Comparacion: CV vs Test
summary = pd.DataFrame({
    'CV ROC-AUC': cv_results['ROC-AUC (val)'],
    'Test ROC-AUC': test_results['ROC-AUC'],
    'CV F1': cv_results['F1 (val)'],
    'Test F1': test_results['F1'],
    'Test Accuracy': test_results['Accuracy'],
})

print('Comparacion CV vs Conjunto de Prueba:')
display(summary.round(4))

mejor_modelo = test_results['ROC-AUC'].idxmax()
mejor_auc    = test_results.loc[mejor_modelo, 'ROC-AUC']

print(f'\nMejor modelo (por ROC-AUC en prueba): {mejor_modelo}')
print(f'ROC-AUC = {mejor_auc:.4f}')

In [ ]:
# Grafica de barras lado a lado: CV vs Test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

model_colors = ['#4C72B0', '#55A868', '#C44E52']

for ax, metric in zip(axes, ['ROC-AUC', 'F1']):
    cv_vals   = summary[f'CV {metric}'].values
    test_vals = summary[f'Test {metric}'].values
    x = np.arange(len(summary))
    w = 0.35

    bars1 = ax.bar(x - w/2, cv_vals,   w, label='Validacion Cruzada',
                   color=model_colors, alpha=0.6)
    bars2 = ax.bar(x + w/2, test_vals, w, label='Conjunto de Prueba',
                   color=model_colors, alpha=0.95)

    ax.set_xticks(x)
    ax.set_xticklabels(summary.index, rotation=12)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'{metric}: CV vs Prueba', fontweight='bold')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)

    for bar, val in zip(bars2, test_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Comparacion Final: Validacion Cruzada vs Conjunto de Prueba',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/comparacion_final.png', bbox_inches='tight')
plt.show()

---
# FASE 6 | CONCLUSIONES Y RECOMENDACIONES
---

## 6.1 Hallazgos principales

### Variable objetivo
- La variable `viralidad` fue construida exitosamente mediante un puntaje compuesto de rango percentil ponderado de las metricas de engagement (views, likes, comments, shares).
- El dataset resulto con un balance de clases aproximadamente 50/50 (corte en la mediana), lo cual simplifica la evaluacion y evita problemas de clases desbalanceadas.

### Desempeno de modelos
- Los tres modelos superaron el umbral minimo definido en los criterios de exito.
- **Gradient Boosting** obtuvo el mejor ROC-AUC en el conjunto de prueba, seguido de **Random Forest**.
- **Regresion Logistica** sirve como referencia interpretable y muestra que existe una senal lineal en los datos.
- La brecha entre el accuracy de entrenamiento y validacion es minima en todos los modelos, indicando ausencia de sobreajuste severo.

### Caracteristicas mas importantes
- Las tasas de engagement relativo (`engagement_rate`, `likes_per_view`, `shares_per_view`) son los predictores mas fuertes.
- El `sentiment_score` y la `duracion_sec` tienen un impacto moderado.
- La `categoria` del contenido y la `region` aportan informacion contextual relevante.

## 6.2 Limitaciones

1. **Construccion de la variable objetivo**: Al usar las mismas metricas para construir `viralidad` y derivar tasas de engagement como caracteristicas, existe una correlacion inherente. Un diseno experimental con datos temporales (predecir viralidad futura con datos del pasado) seria mas riguroso.
2. **Muestra**: Se uso una muestra de 100,000 registros del dataset de 1,000,000 por eficiencia computacional. El modelo completo podria explotar mas patrones.
3. **Hiperparametros**: No se realizo busqueda exhaustiva de hiperparametros (GridSearch/RandomSearch). Optimizarlos podria mejorar el desempeno.
4. **Variables externas**: Factores como tendencias del momento, algoritmo de recomendacion de YouTube y campanas de marketing no estan capturados en el dataset.

## 6.3 Recomendaciones

1. **Para creadores de contenido**: Priorizar videos con alta tasa de interaccion (likes/shares por vista) sobre el volumen bruto de vistas.
2. **Mejora del modelo**: Explorar modelos como XGBoost/LightGBM para mayor eficiencia, y aplicar SHAP values para interpretabilidad local.
3. **Validacion temporal**: Usar los primeros meses del dataset para entrenar y los ultimos para validar, simulando un escenario real de prediccion.
4. **Dashboard**: Los resultados del modelo y los hallazgos del EDA estan disponibles para su visualizacion en Power BI/Tableau a traves del archivo `data/processed/data_processed.csv`.

In [ ]:
# Resumen ejecutivo final
print('='*60)
print('      RESUMEN EJECUTIVO - PROYECTO VIRALIDAD YOUTUBE')
print('='*60)
print(f'Dataset        : {SAMPLE_SIZE:,} videos muestreados de 1,000,000')
print(f'Variable obj   : viralidad (binaria, balance 50/50)')
print(f'Caracteristicas: {X_train_proc.shape[1]} (tras OHE y escalado)')
print(f'Particion      : 80% entrenamiento / 20% prueba (estratificada)')
print(f'Validacion     : StratifiedKFold (5 folds)')
print()
print('Metricas en conjunto de prueba:')
print(test_results[['Accuracy','F1','ROC-AUC']].round(4).to_string())
print()
print(f'Mejor modelo   : {mejor_modelo} (ROC-AUC = {mejor_auc:.4f})')
print('='*60)